# Context / gate data preparation
Current 30-clip panel is DEV only. Gate training needs independent TRAIN audio with validated timestamps. This notebook audits local prerequisites; it does not pretend a gate training dataset already exists.

In [1]:
from pathlib import Path
import sys, json
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Open notebook from the project workspace'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
DATA = PROJECT/'data'
RAW_DATA = PROJECT.parent/'dataset/ASR'
assert PYTHON.is_file(), 'Run ASR/local/setup.ps1 first'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
RUNNER = PROJECT/'scripts/local_run.py'
print('Project:', PROJECT)
print('Python:', PYTHON)
print('Ready. Run the specific task cell; model jobs do not start in this setup cell.')


Project: D:\1.FPT_FALL26\DPL - Deep Learning\ASR
Python: D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe
Ready. Run the specific task cell; model jobs do not start in this setup cell.


In [2]:
import pandas as pd
from urllib.parse import urlparse, parse_qs
meta = RAW_DATA/'ViMedCSS/ViMedCSS-Metadata'
train = pd.read_csv(meta/'train_set.csv')
dev = pd.read_csv(meta/'valid_set.csv')
def vid(url):
    p = urlparse(str(url))
    return p.path.strip('/') if p.hostname in ['youtu.be','www.youtu.be'] else parse_qs(p.query).get('v',[''])[0]
def seconds(value):
    total = 0.0
    for part in str(value).split(':'): total = total*60+float(part)
    return total
# Conservative separation from ALL official DEV source videos.
dev_ids = set(dev.original_video_link.map(vid))
train['video_id'] = train.original_video_link.map(vid)
train['start_s'] = train.start_time.map(seconds)
train['end_s'] = train.end_time.map(seconds)
train_panel = train[train.video_id.ne('') & ~train.video_id.isin(dev_ids) & train.start_s.ge(20) & (train.end_s-train.start_s).gt(0) & (train.end_s-train.start_s).le(30)].copy()
train_panel['alignment_status'] = 'pending'
out = PROJECT/'local/context/train_candidates.csv'
out.parent.mkdir(parents=True, exist_ok=True)
train_panel.sort_values(['video_id','start_s']).to_csv(out,index=False)
print('TRAIN candidates:', len(train_panel), '| Videos:', train_panel.video_id.nunique())
print('Saved:',out)
display(train_panel.head())

TRAIN candidates: 3899 | Videos: 1276
Saved: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\context\train_candidates.csv


,segment_id,duration_seconds,segment_text,cs_terms_list,cs_terms_count,topic,original_video_link,original_video_title,start_time,end_time,video_id,start_s,end_s,alignment_status
1,Med_CS-0-10,11,Mặc dù chúng không thể chuyển thành dihydrotes...,testosteron,1,Medical Sciences,https://www.youtube.com/watch?v=GUEt9JwUd-4,Thiếu hụt 5 alpha reductase - cơ quan sd lưỡng...,03:42,03:52,GUEt9JwUd-4,222.0,232.0,pending
2,Med_CS-0-11,9,"Ngoài ra, lượng testosterone cao còn làm trầm ...",testosteron,1,Medical Sciences,https://www.youtube.com/watch?v=GUEt9JwUd-4,Thiếu hụt 5 alpha reductase - cơ quan sd lưỡng...,03:52,04:00,GUEt9JwUd-4,232.0,240.0,pending
3,Med_CS-0-12,8,Chẩn đoán thiếu 5 alpha reductase thường dựa v...,reductase,1,Medical Sciences,https://www.youtube.com/watch?v=GUEt9JwUd-4,Thiếu hụt 5 alpha reductase - cơ quan sd lưỡng...,04:14,04:21,GUEt9JwUd-4,254.0,261.0,pending
4,Med_CS-0-14,12,Các nghiên cứu trong phòng lab thường cho thấy...,testosteron,1,Medical Sciences,https://www.youtube.com/watch?v=GUEt9JwUd-4,Thiếu hụt 5 alpha reductase - cơ quan sd lưỡng...,04:31,04:42,GUEt9JwUd-4,271.0,282.0,pending
5,Med_CS-0-16,11,"Nói tóm lại, thiếu hụt 5 alpha reductase sẽ là...",reductase; testosteron,2,Medical Sciences,https://www.youtube.com/watch?v=GUEt9JwUd-4,Thiếu hụt 5 alpha reductase - cơ quan sd lưỡng...,05:08,05:18,GUEt9JwUd-4,308.0,318.0,pending


## Next gate stage
Download/validate TRAIN original audio, generate reference-blind KEEP/context candidates, then create labels from TRAIN transcripts. Split gate TRAIN/CAL by source video. Never use reference WER, gold transcript, or oracle choice as input features. Export features once; small gate training can use CPU. First verify inference on local V1 in notebook 03. History 5s anomaly and EOS/truncation audit remain unresolved; do not freeze a history policy yet.